# Do the text representations improve the same predictor?

This is the downstream scientific analysis, separate from extraction agreement and runtime checks. It reads only complete, checksum-validated saved CV arms. **No provider, model fit, benchmark job or statistical test runs here.** With no completed benchmark, the notebook prints that status and creates no apparent effect sizes.

## 1. Campaign identity and complete-data coverage

Every paired contrast shares dataset, predictor, checkpoint, split and outer fold. Incomplete arms remain reported and cannot contribute to a five-fold effect. All recorded metrics and immutable prediction references are retained in the final report.


In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import matplotlib.pyplot as plt
from src.evaluation.results import collect
from src.evaluation.notebook_evidence import saved_pca_diagnostics
from src.utils.notebook_display import display_local, display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.visualize import figures, style
from src.visualize.paired import paired_effect_figures, embedding_effect_figures, pca_variance_figures
style.apply()
NOTEBOOK = "4. experiment_1/01_paired_representation_effects"
begin_report()
report_section("1. Campaign provenance and complete-data coverage")
display_json("Notebook execution identity", provenance())
status, folds, per_dataset, paired = collect()
display_json("Saved benchmark coverage", status)
figures.clear(NOTEBOOK)
save = None
record(folds, "All fold-level recorded metrics, timings and prediction references")
if folds.empty:
    print("Pending: no completed downstream folds. Runtime contracts and pilots are not predictive evidence.")


Pending: no completed downstream folds. Runtime contracts and pilots are not predictive evidence.


## 2. Paired representation effects

The plots hold the predictor fixed and show per-dataset five-fold mean differences for joint, per-column and combined decision features. Dataset points remain visible beside their mean; CV fold variation is not a confidence interval. Positive values mean improvement, with metric direction stated on each axis. AUC, multiclass log loss and RMSE are descriptive views of logged outcomes, **not selection or approval of a confirmatory endpoint**. Every other logged metric remains in the full tables. Presence controls are plotted as separate named references when available.


In [2]:
report_section("2. Paired effects within fixed predictors")
display_table("All complete-arm per-dataset metrics", per_dataset, preview_rows=8)
display_table("All matched outer-fold differences", paired, preview_rows=8)
from itertools import chain
for fig, name, caption in chain(paired_effect_figures(paired), embedding_effect_figures(paired)):
    if save is None:
        save = figures.FigureSaver(NOTEBOOK)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 3. What information does PCA retain?

For each frozen encoder, cumulative explained variance is read from actual outer-training PCA diagnostics. Joint curves and per-column PCA32 points use separate panels. Neither validation nor test rows fit PCA. Variance fractions are descriptions of each representation, not predictive relevance or a comparable amount of task information across encoders. Requested/fitted ranks and training counts remain in the table; no curve is invented before fitted artifacts exist.


In [3]:
report_section("3. Training-only PCA diagnostics")
pca = saved_pca_diagnostics(folds)
display_table("Observed PCA rank, training coverage and explained variance", pca, preview_rows=8)
for fig, name, caption in pca_variance_figures(pca):
    if save is None:
        save = figures.FigureSaver(NOTEBOOK)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)


## 4. Original-text reference systems

TabSTAR, ConTextTab and AG-MM change the predictor and text handling together. Their complete results are descriptive system references; they are not a within-predictor representation contrast. No best model, winning combination, confirmatory test or metric is chosen here.


In [4]:
report_section("4. Native-text references and scientific boundaries")
if not per_dataset.empty:
    display_table("Original-text system references", per_dataset[per_dataset.feature_set.eq("original_features")])
record("Dataset is the comparison unit after averaging paired folds. Missing or unsupported arms are not zero performance. "
       "No hypothesis test, outcome-based method selection or current-code runtime acceptance is performed.", "Interpretation boundary")


## 5. Decision features versus frozen embeddings

Every decision recipe is directly paired with every embedding recipe in the **same predictor, dataset, checkpoint and fold**. No winning recipe is selected. These panels answer the central representation question more directly than separate gains over no-text. J8–J256 mean joint PCA dimensions; P32 means 32 dimensions per text column. Cells with incomplete evidence stay blank. Tables retain the datasets/folds behind each cell so coverage differences cannot be mistaken for representation effects.


In [5]:
report_section("5. Direct decision-versus-embedding comparisons")
from src.evaluation.representation_views import decision_vs_embedding, measured_extraction
from src.visualize.representation_views import direct_comparison_figures, control_figures, overlap_figures
contrasts = decision_vs_embedding(folds)
display_table("All direct matched recipe comparisons", contrasts, preview_rows=8)
for fig, name, caption in direct_comparison_figures(contrasts):
    if save is None:
        save = figures.FigureSaver(NOTEBOOK)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)
if contrasts.empty:
    print("Direct comparisons await complete decision and embedding arms on matching folds.")


Direct comparisons await complete decision and embedding arms on matching folds.


## 6. Controls and previously seen text

The controls separate semantic information from text presence and simple properties. The overlap panels ask whether gains remain when the training fold contains neither the same joint text nor any of the individual text fields. **All-columns-unseen is a subset of unseen-joint**, not an additional disjoint population. We retain the existing splits and thresholds; these conditional diagnostics are not causal effects and cannot rule out pretraining contamination. All cohort counts and undefined metrics remain in the report. Post-selection exploratory combinations are excluded from the control panel.


In [6]:
report_section("6. Controls and conditional text-overlap effects")
for fig, name, caption in chain(control_figures(paired), overlap_figures(paired)):
    if save is None:
        save = figures.FigureSaver(NOTEBOOK)
    save(fig, name, caption=caption)
    display_local(fig)
    plt.close(fig)
if not folds.empty:
    display_table("Text-overlap cohort sizes and all conditional metrics", folds[folds.metric.str.startswith("text.")], preview_rows=8)


## 7. Measured computational burden and feature width

The records below distinguish extraction time, predictor fitting/prediction time and representation width. Width is measured **after PCA, before predictor-specific categorical preprocessing**; missing legacy measurements stay missing. Extraction summaries retain dataset, run and hardware identity. Pilot timings are excluded. Partial runs, retries and cache reuse must be reviewed before linking their time to a completed feature artifact; we do not present them as a full-extraction price or invent a GPU-to-dollar conversion. The stored timing journals allow that linkage once the actual extraction campaign is complete.


In [7]:
report_section("7. Observed time, representation width and extraction coverage")
if not folds.empty:
    burden = folds[folds.metric.str.startswith(("seconds.", "columns.", "resources."))]
    display_table("Per-fold timing, representation width and resources", burden, preview_rows=12)
extraction_times = measured_extraction()
display_table("Measured full-extraction stages with run and hardware identity", extraction_times, preview_rows=12)
if extraction_times.empty:
    print("Full-extraction timing evidence is pending; no price-performance claim is made.")


Full-extraction timing evidence is pending; no price-performance claim is made.


## Complete analysis report

All displayed tables, plotted values and source identities are printed below and saved in `output_JEVPFN/All Results.md`.


In [8]:
finish_report(NOTEBOOK, f"Coverage: {status}. No fitting or inference. " + (save.summary() if save is not None else "No figures: complete predictive records are pending."))


4. experiment_1/01_paired_representation_effects

1. Campaign provenance and complete-data coverage


Notebook execution identity
{
  "code_sha256": "cabb9dfa5466d09b8390a85083abf383a89f65321aacb9365d5a27bf1092c168",
  "git_head": "60470462a8c71c6782a10a4001eb48db3391710d",
  "git_dirty": true,
  "tfm_library_pin": "5f1b3f8f1611681579aba052a86c2626a6a17b96 dm-library (remotes/origin/HEAD)\n 702d9ab0b66f6639ba8d443ff9c4a91231964465 tfm-library (remotes/origin/HEAD)\n 3bcdf55e576d10161d6189b40ac39380bbba9eca vsc-library (remotes/origin/HEAD)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.3",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Saved benchmark coverage
{
  "status": "no_completed_folds"
}


All fold-level recorded metrics, timings and prediction references
Empty DataFrame
Columns: []
Index: []


2. Paired effects within fixed predictors


All complete-arm per-dataset metrics
Empty DataFrame
Columns: []
